# Формирование датасета

In [8]:
# Установка библиотек (запустите один раз)
!pip install -r requirements.txt

import os
import cv2
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

import kagglehub

from tqdm import tqdm

# Для воспроизводимости результатов
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Проверка GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.6/555.6 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.0 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.7-py3-none-any.whl size=53345 sha256=45d0378827212d65c25699ef8cf00db45db0d184353f4af93b1b8f842a3ef47e
  Stored in directory: /root/.cache/pip/wheels/91/36/f0/c2367865493d51b1336976

In [9]:
# Скачивание датасета через kagglehub
# ВАЖНО: если это ваш первый запуск, Colab попросит авторизоваться через Kaggle API token
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
base_dir = Path(path) / "chest_xray"

print(f"Датасет сохранён в: {base_dir}")
print("Структура папок:")
for p in base_dir.iterdir():
    if p.is_dir():
        print(f"  {p.name}/")

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Датасет сохранён в: /kaggle/input/chest-xray-pneumonia/chest_xray
Структура папок:
  chest_xray/
  __MACOSX/
  val/
  test/
  train/


### Разбиение на выборки

In [10]:
# Собираем ВСЕ изображения из всех папок датасета
data = []

for split_name in ["train", "val", "test"]:
    split_dir = base_dir / split_name

    # Класс NORMAL
    normal_dir = split_dir / "NORMAL"
    if normal_dir.exists():
        for img_path in normal_dir.glob("*.jpeg"):
            data.append((str(img_path), "NORMAL"))

    # Класс PNEUMONIA -> разделяем на BACTERIAL и VIRAL по имени файла
    pneumonia_dir = split_dir / "PNEUMONIA"
    if pneumonia_dir.exists():
        for img_path in pneumonia_dir.glob("*.jpeg"):
            name = img_path.name.lower()
            if "bacteria" in name:
                data.append((str(img_path), "BACTERIAL"))
            elif "virus" in name:
                data.append((str(img_path), "VIRAL"))
            else:
                data.append((str(img_path), "PNEUMONIA_OTHER"))

df = pd.DataFrame(data, columns=["path", "class"])

print(f"Всего изображений: {len(df)}")
print("\nРаспределение по классам:")
print(df["class"].value_counts())

# Проверяем, есть ли класс PNEUMONIA_OTHER
if "PNEUMONIA_OTHER" in df["class"].values:
    other_count = (df["class"] == "PNEUMONIA_OTHER").sum()
    print(f"\n⚠ ВНИМАНИЕ: {other_count} файлов не содержат 'bacteria' или 'virus' в имени!")
    print("Примеры таких файлов:")
    others = df[df["class"] == "PNEUMONIA_OTHER"]["path"].head(5)
    for p in others:
        print(f"  {Path(p).name}")

# Стратифицированное разбиение на 4 части: 50% train, 15% val, 20% cal, 15% test
X = df["path"].values
y = df["class"].values

# Шаг 1: отделяем train (50%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42
)

# Шаг 2: из temp (50%) отделяем val (30% от temp = 15% от общего)
X_val, X_temp2, y_val, y_temp2 = train_test_split(
    X_temp, y_temp, test_size=0.7, stratify=y_temp, random_state=42
)

# Шаг 3: из temp2 (35%) делим на cal (57.14% от temp2 = 20% от общего) и test (42.86% от temp2 = 15% от общего)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp2, y_temp2, test_size=0.4286, stratify=y_temp2, random_state=42
)

print(f"\nРазмеры выборок:")
print(f"  Train:       {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation:  {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Calibration: {len(X_cal)} ({len(X_cal)/len(X)*100:.1f}%)")
print(f"  Test:        {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nРаспределение классов в Train:")
print(Counter(y_train))
print(f"\nРаспределение классов в Validation:")
print(Counter(y_val))
print(f"\nРаспределение классов в Calibration:")
print(Counter(y_cal))
print(f"\nРаспределение классов в Test:")
print(Counter(y_test))

Всего изображений: 5856

Распределение по классам:
class
BACTERIAL    2780
NORMAL       1583
VIRAL        1493
Name: count, dtype: int64

Размеры выборок:
  Train:       2928 (50.0%)
  Validation:  878 (15.0%)
  Calibration: 1171 (20.0%)
  Test:        879 (15.0%)

Распределение классов в Train:
Counter({'BACTERIAL': 1390, 'NORMAL': 791, 'VIRAL': 747})

Распределение классов в Validation:
Counter({'BACTERIAL': 417, 'NORMAL': 237, 'VIRAL': 224})

Распределение классов в Calibration:
Counter({'BACTERIAL': 556, 'NORMAL': 317, 'VIRAL': 298})

Распределение классов в Test:
Counter({'BACTERIAL': 417, 'NORMAL': 238, 'VIRAL': 224})


### Препроцессинг изображений с ленивым чтением. Класс Dataset

In [11]:
class ChestXRayDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = paths
        self.label_map = {"NORMAL": 0, "BACTERIAL": 1, "VIRAL": 2}
        self.labels = [self.label_map[l] for l in labels]
        self.augment = augment

        self.base_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        if self.augment:
            self.aug_transform = transforms.Compose([
                transforms.RandomRotation(15),
                transforms.RandomHorizontalFlip(p=0.5),
            ])
        else:
            self.aug_transform = None

        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx], cv2.IMREAD_GRAYSCALE)
        img = self.clahe.apply(img)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = self.base_transform(img)

        if self.aug_transform is not None:
            img = self.aug_transform(img)

        label = self.labels[idx]
        return img, label

# Создаём объекты Dataset - теперь 4 штуки
train_dataset = ChestXRayDataset(X_train, y_train, augment=True)
val_dataset = ChestXRayDataset(X_val, y_val, augment=False)  # НОВАЯ выборка
cal_dataset = ChestXRayDataset(X_cal, y_cal, augment=False)
test_dataset = ChestXRayDataset(X_test, y_test, augment=False)

print(f"Train dataset: {len(train_dataset)} изображений")
print(f"Validation dataset: {len(val_dataset)} изображений")
print(f"Calibration dataset: {len(cal_dataset)} изображений")
print(f"Test dataset: {len(test_dataset)} изображений")

Train dataset: 2928 изображений
Validation dataset: 878 изображений
Calibration dataset: 1171 изображений
Test dataset: 879 изображений


### Подгрузчики изображений для каждой подгруппы (выборки)

In [12]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(  # НОВЫЙ DataLoader
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
cal_loader = DataLoader(
    cal_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print("DataLoaders созданы успешно")

DataLoaders созданы успешно


# Формирование csv

In [13]:
from safetensors.torch import load_file

In [14]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Создаём папку outputs, если её нет
os.makedirs("outputs", exist_ok=True)

# === 1. Загрузка модели ===
print("Загрузка модели...")
model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)

from safetensors.torch import load_file
state_dict = load_file("best_model.safetensors")
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()
print(f"Модель загружена на устройство: {device}")

# === 2. Модифицируем Dataset для возврата имён файлов ===
class ChestXRayDatasetWithNames(ChestXRayDataset):
    def __getitem__(self, idx):
        img, label = super().__getitem__(idx)
        filename = Path(self.paths[idx]).name
        return img, label, filename

# Пересоздаём датасеты с возвратом имён
cal_dataset_names = ChestXRayDatasetWithNames(X_cal, y_cal, augment=False)
test_dataset_names = ChestXRayDatasetWithNames(X_test, y_test, augment=False)

cal_loader_names = DataLoader(
    cal_dataset_names, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader_names = DataLoader(
    test_dataset_names, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

# === 3. Функция инференса ===
def run_inference(model, dataloader, split_name):
    """Прогоняет модель и возвращает список словарей по контракту"""
    results = []

    with torch.no_grad():
        for imgs, labels, filenames in tqdm(dataloader, desc=f"Инференс {split_name}"):
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1).cpu().numpy()

            for i in range(len(filenames)):
                results.append({
                    'image_id': filenames[i],
                    'split': split_name,
                    'true_label': labels[i].item(),
                    'prob_normal': float(probs[i, 0]),
                    'prob_bacterial': float(probs[i, 1]),
                    'prob_viral': float(probs[i, 2]),
                })

    return results

# === 4. Запуск инференса ===
print("\nЗапуск инференса на калибровочной выборке...")
cal_results = run_inference(model, cal_loader_names, "calib")

print("\nЗапуск инференса на тестовой выборке...")
test_results = run_inference(model, test_loader_names, "test")

# === 5. Объединение и валидация ===
all_results = cal_results + test_results
df = pd.DataFrame(all_results)

# Валидация: сумма вероятностей должна быть ≈ 1.0
df['prob_sum'] = df['prob_normal'] + df['prob_bacterial'] + df['prob_viral']
max_deviation = (df['prob_sum'] - 1.0).abs().max()

print(f"\n=== Валидация распределения ===")
print(f"Максимальное отклонение суммы от 1.0: {max_deviation:.2e}")
if max_deviation < 1e-5:
    print("✅ Валидация пройдена: сумма вероятностей == 1.0")
else:
    print("⚠ ВНИМАНИЕ: сумма вероятностей отклоняется от 1.0!")

# Удаляем вспомогательную колонку перед сохранением
df = df.drop(columns=['prob_sum'])

# === 6. Сохранение ===
output_path = "outputs/predictions.csv"
df.to_csv(output_path, index=False)

print(f"\n✅ Инференс завершён!")
print(f"📊 Всего обработано изображений: {len(df)}")
print(f"   - Калибровочная выборка (calib): {len(cal_results)}")
print(f"   - Тестовая выборка (test): {len(test_results)}")
print(f"💾 Файл сохранён: {output_path}")

# Показываем первые строки
print("\nПример данных:")
print(df.head(10).to_string(index=False))

Загрузка модели...
Модель загружена на устройство: cuda

Запуск инференса на калибровочной выборке...


Инференс calib: 100%|██████████| 74/74 [00:35<00:00,  2.08it/s]



Запуск инференса на тестовой выборке...


Инференс test: 100%|██████████| 55/55 [00:23<00:00,  2.37it/s]


=== Валидация распределения ===
Максимальное отклонение суммы от 1.0: 1.19e-07
✅ Валидация пройдена: сумма вероятностей == 1.0

✅ Инференс завершён!
📊 Всего обработано изображений: 2050
   - Калибровочная выборка (calib): 1171
   - Тестовая выборка (test): 879
💾 Файл сохранён: outputs/predictions.csv

Пример данных:
                     image_id split  true_label  prob_normal  prob_bacterial  prob_viral
  person147_bacteria_706.jpeg calib           1     0.030785        0.693837    0.275379
   person91_bacteria_449.jpeg calib           1     0.094712        0.761414    0.143874
person1944_bacteria_4869.jpeg calib           1     0.000688        0.008789    0.990523
     person351_virus_726.jpeg calib           2     0.000112        0.043531    0.956358
   person1672_virus_2888.jpeg calib           2     0.185030        0.156790    0.658180
    NORMAL2-IM-0525-0001.jpeg calib           0     0.964620        0.009717    0.025663
 person599_bacteria_2455.jpeg calib           1     0.2044